<div dir="rtl">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/Computer_vision_Arabic-/blob/main/labs/03_custom_training.ipynb)


# 🎓 أساسيات YOLO الجزء 3: التدريب المخصص (Custom Training) 🏗️

مرحباً بك في المحطة الثالثة! بعد أن تعلمنا كيف نستخدم النماذج الجاهزة، حان الوقت لنتعلم كيف نصنع "عقل" خاص بنا يفهم بياناتنا الخاصة. 

### 🤔 لماذا نحتاج للتدريب المخصص؟
النماذج الجاهزة (مثل التي تدربت على COCO) تعرف كيف تكتشف الكراسي والسيارات والكلاب.. لكن ماذا لو كنت تريد اكتشاف **نوع معين من البراغي** في مصنع؟ أو **مرض معين** في أوراق الشجر؟ هنا نحتاج لعملية تسمى **الضبط الدقيق (Fine-tuning)**.

--- 

## 🛠️ 1. تجهيز بيئة العمل

نتأكد أولاً من تثبيت مكتبة `ultralytics` وجاهزية الأدوات. 

> **تلميح:** إذا كنت تعمل على Google Colab، تأكد من تغيير نوع الجهاز (Runtime Type) إلى **T4 GPU** ليكون التدريب أسرع.

</div>

In [ ]:
# تثبيت مكتبة Ultralytics
%pip install -q ultralytics
import ultralytics

# التحقق من جاهزية البيئة والمعالج الرسومي (GPU)
# الـ GPU هو "محرك التوربو" الذي يجعل التدريب أسرع بـ 10-50 مرة من المعالج العادي.
ultralytics.checks()

<div dir="rtl">

## 📦 2. مفهوم الـ Fine-tuning ومجموعة بيانات COCO8

### 🧠 تشبيه بسيط: نقل المعرفة
تخيل أنك تريد تعليم شخص قيادة الشاحنات. هل تبدأ معه من الصفر لتعلمه كيف يرى الطريق ويستخدم المكابح؟ لا! بل تأتي بشخص يعرف قيادة السيارات العادية (النموذج المدرب مسبقاً) وتعلمه فقط الفروقات البسيطة للشاحنة. هذا يوفر وقتاً وجهداً هائلاً.

### 🗺️ كيف يعرف YOLO البيانات؟ (ملف الـ YAML)
ملف الإعدادات (مثل `coco8.yaml`) هو بمثابة "الدليل الإرشادي". بدونه، لا يعرف YOLO أين الصور وما هي الأسماء. يحتوي الملف عادة على:
*   `path`: المسار الرئيسي للبيانات.
*   `train`: صور التدريب (التي يذاكر منها).
*   `val`: صور الاختبار (التي يختبر نفسه بها).
*   `names`: قائمة بالأسماء (0: شخص، 1: دراجة، إلخ).

</div>

In [ ]:
from ultralytics import YOLO

# 1. تحميل نموذج YOLO26n (عقل ذكي مسبقاً يعرف الأشكال الأساسية)
model = YOLO("yolo26n.pt")

# 2. بدء عملية التدريب
# 💡 شرح المعاملات (Arguments):
# data: ملف الإعدادات (الخريطة).
# epochs: تخيل الـ Epoch كأنه "فصل دراسي". كلما زاد عدد الفصول، زادت خبرة الطالب بالمنهج.
# imgsz: دقة الصور المدخلة. الرقم 640 هو المعيار الذهبي للتوازن بين الدقة والسرعة.
model.train(data="coco8.yaml", epochs=3, imgsz=640)

<div dir="rtl">

### 💡 تمرين: المحسنات (Optimizers)

**تحدي:** حاول تعديل الكود ليستخدم محسن `AdamW`. المحسن هو الخبير الرياضي الذي يوجه النموذج للوصول لأفضل دقة بأقصر طريق. جرب وشاهد هل يتحسن الـ Loss (الخطأ) بسرعة أكبر؟

</div>

In [ ]:
# قم بتجربة التدريب هنا بإعدادات مختلفة
# model.train(data="coco8.yaml", epochs=3, imgsz=640, optimizer='AdamW')


<div dir="rtl">

## 🧪 3. تشغيل عملية التحقق (Validation)

بعد أن أنهى النموذج "دراسته" (التدريب)، حان وقت الاختبار. نستخدم بيانات "مستقلة" لم يراها النموذج أثناء التدريب للتأكد من أنه لم يحفظ الصور حفظاً بل فهم القواعد.

### 📊 فك شفرات لغة المقاييس:
1.  **Precision (الدقة)**: لو أشار النموذج لـ 10 أجسام بأنها "قطة" وكان منها 9 قطط فعلاً، فالدقة 90%.
2.  **Recall (الاستدعاء)**: لو كان في الغرفة 10 قطط واكتشف النموذج 7 فقط، فالاستدعاء 70%.
3.  **mAP50**: هو المقياس الأهم. يخبرنا بمدى جودة المربعات المرسومة وصحة التصنيف عند تقاطع بنسبة 50%.

</div>

In [ ]:
# 1. تحميل "أفضل" نسخة (best.pt) - هذه هي النسخة التي حصلت على أعلى درجات في الاختبارات أثناء التدريب
model = YOLO("runs/detect/train/weights/best.pt")

# 2. تشغيل عملية التحقق
metrics = model.val(data="coco8.yaml")

# 3. طباعة مقاييس الأداء الأساسية
print(f"\nmAP@0.5 (الدرجة العامة): {metrics.box.map50:.4f}")
print(f"Precision (الدقة):      {metrics.box.mp:.4f}")
print(f"Recall (الاستدعاء):     {metrics.box.mr:.4f}")

<div dir="rtl">

## ⚖️ 4. المقارنة والتحليل العلمي

لماذا نتعب أنفسنا بالتدريب المخصص؟ 

| النموذج | متى نستخدمه؟ |
| :--- | :--- |
| **Pretrained (الجاهز)** | عندما تكون الكائنات عامة جداً (أشخاص، كراسي، كلاب). |
| **Fine-tuned (المخصص)** | عندما نحتاج لدقة عالية في بيئة محددة أو كائنات لم يسبق للنموذج رؤيتها. |

**تنبيه:** التدريب على 8 صور فقط (COCO8) هو مجرد "تسخين". في الواقع، ستحتاج لمئات أو آلاف الصور لكل فئة لتحصل على نموذج يمكن الاعتماد عليه في مشروع تجاري.

</div>

In [ ]:
# تحميل النموذج الأصلي الخام للمقارنة
pretrained = YOLO("yolo26n.pt")
pretrained_metrics = pretrained.val(data="coco8.yaml")

print("📊 لوحة مقارنة الأداء:")
print(f"  النموذج الأصلي (Pretrained): {pretrained_metrics.box.map50:.4f}")
print(f"  النموذج بعد الضبط (Fine-tuned): {metrics.box.map50:.4f}")

<div dir="rtl">

## 📂 كنز النتائج: مجلد `runs/` 
بعد كل عملية تدريب، يقوم YOLO بإنشاء تقرير كامل. افتح المجلدات على اليسار وابحث عن `runs/detect/train/`:
*   `results.png`: يظهر لك منحنى التعلم. إذا كان الخط ينزل للأسفل، فهذا يعني أن النموذج يتحسن.
*   `F1_curve.png`: يوضح التوازن بين الدقة والاستدعاء.
*   `val_batch0_pred.jpg`: صورة حقيقية وعليها توقعات النموذج، لترى بعينك مدى جودته.

---
## 🎉 مبروك! 
لقد أتممت الآن الجزء الأصعب في رحلة الذكاء الاصطناعي: **تعليم الآلة**. في المعامل القادمة، سنتعلم كيف نقيم النماذج من منظور تجاري ومالي.

</div>